# 00 — Data Sanity Checks

**Purpose:** verify that the raw data pipeline in `src/data.py` is correct *before* any training happens.

**Why this notebook exists:** roughly 80% of "the model isn't learning" bugs trace back to the data layer — a chromosome-name mismatch, an off-by-one in window extraction, or a leaky train/test split. Those bugs are silent: the model trains, the loss goes down, and the AUROC looks plausible, but the model has learned the wrong thing. This notebook is the checklist that catches them.

**Prerequisites:**
- `data/raw/creb1_peaks.bed` (ENCODE narrowPeak, GRCh38)
- `data/raw/Homo_sapiens.GRCh38.dna.primary_assembly.fa` (Ensembl GRCh38)

If either is missing, run `bash scripts/download_data.sh` from the repo root first.

**When to run:** after any change to `src/data.py`, and once on a fresh clone.

In [1]:
from src.notebook_setup import set_repo_root
set_repo_root()

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
from pyfaidx import Fasta

from src.data import (
    CANONICAL_CHROMS, DataConfig, build_dataset,
    load_peaks, one_hot, extract_window, _peak_center,
)

## 1. The BED file looks sane

We expect a narrowPeak file with 10 columns, coordinates in GRCh38, and chromosome names like `chr1` (which `load_peaks` normalizes to `1`).

**What to verify in the output below:**
- `chrom` column is `object` (string), not `int64`
- Only canonical chromosomes appear (`1`–`22`, `X`, `Y`) — no `chrUn_*`, no alt contigs, no `MT`
- `summit >= peak length` is **0** (a nonzero value means the summit falls outside the peak interval, which is malformed narrowPeak)

In [7]:
cfg = DataConfig()
peaks = load_peaks(cfg.bed_path)

print(f"peaks: {len(peaks):,}")
print(f"columns: {list(peaks.columns)}")
print(f"chrom dtype: {peaks.chrom.dtype}")
print(f"start dtype: {peaks.start.dtype}")
display(peaks.head())
print("\nper-chromosome counts:")
display(peaks.chrom.value_counts().sort_index())

print(f"\nsummit == -1 rows: {(peaks.summit == -1).sum():,}")
print(f"summit >= peak length (bad): {(peaks.summit >= (peaks.end - peaks.start)).sum():,}")

peaks: 45,725
columns: ['chrom', 'start', 'end', 'signalValue', 'summit']
chrom dtype: str
start dtype: int64


,chrom,start,end,signalValue,summit
0,11,18802836,18803226,15.15450,195
1,16,11310185,11310575,15.18880,195
2,18,37338414,37338804,15.20475,195
3,20,35542044,35542434,15.22287,195
4,19,41439680,41440070,15.31159,195



per-chromosome counts:


chrom
1     3255
10    2003
11    1846
12    1863
13    1107
14    1983
15    1074
16    2703
17    1941
18    1011
19    1620
2     5233
20    2387
21     520
22     800
3     2444
4     2094
5     2115
6     3225
7     2198
8     1952
9     1548
X      743
Y       60
Name: count, dtype: int64


summit == -1 rows: 0
summit >= peak length (bad): 6


## 2. The FASTA and BED agree on chromosome names

This is the check that catches the single most common data bug in this pipeline: the BED uses `chr1` and the FASTA uses `1` (or vice versa). `load_peaks` strips `chr`, but if the FASTA headers have a different convention (e.g. `chr1` vs `1` vs `NC_000001.11`), every `extract_window` call will `KeyError` — or worse, silently return empty sequences.

**What to verify below:**
- `in BED but not FASTA` must be **empty**
- `chr1 length` should be ~248,956,422 bp for GRCh38 (if it's ~3M, you downloaded the wrong file)
- The first 60 bp of chr1 should start with `N`s — GRCh38 chr1 begins with ~10,000 `N`s (telomeric repeat). If it starts with clean `ACGT`, you're looking at the wrong assembly or a masked version.

In [6]:
fa = Fasta(str(cfg.fasta_path), as_raw=True)

fa_keys = set(fa.keys())
bed_keys = set(peaks.chrom.unique())

print(f"FASTA sequences: {len(fa_keys):,}")
print(f"BED chroms after filter: {sorted(bed_keys)}")
print(f"in BED but not FASTA: {bed_keys - fa_keys}")
print(f"canonical chroms missing from FASTA: {[c for c in CANONICAL_CHROMS if c not in fa_keys]}")

print(f"\nchr1 length: {len(fa['1']):,} bp")
print(f"first 60 bp of chr1: {str(fa['1'][1:61]).upper()}")

FASTA sequences: 194
BED chroms after filter: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '3', '4', '5', '6', '7', '8', '9', 'X', 'Y']
in BED but not FASTA: set()
canonical chroms missing from FASTA: []

chr1 length: 248,956,422 bp
first 60 bp of chr1: NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN


## 3. Window extraction produces the right bases

`extract_window` is where the BED↔pyfaidx 0-based/1-based conversion happens. An off-by-one here corrupts every positive sample and you won't notice until the model underperforms. This cell prints a real window so you can eyeball it.

**What to verify below:**
- `bases covered` should be **500 / 500** for an interior peak (no clipping). If less, the peak is near a chromosome edge.
- The reconstructed sequence should look like DNA — if it's mostly `N`s, either the peak is in a centromere/telomere, or the offset is wrong.
- The sequence should correspond to what you'd see if you looked up the same coordinates in UCSC Genome Browser (spot-check one if you're paranoid).

In [8]:
row = peaks.iloc[0]
center = _peak_center(row)
x = extract_window(fa, row.chrom, center, cfg.window)

print(f"peak: {row.chrom}:{row.start}-{row.end}  summit={row.summit}  center={center}")
print(f"window shape: {x.shape}   sum per base: {x.sum(axis=0)[:5]} ...")
print(f"bases covered (nonzero columns): {(x.sum(axis=0) > 0).sum()} / {cfg.window}")

seq = "".join("ACGT"[int(np.argmax(x[:, i]))] if x[:, i].any() else "N"
              for i in range(x.shape[1]))
print(f"\nreconstructed (first 80): {seq[:80]}")
print(f"N count in window: {seq.count('N')}")

peak: 11:18802836-18803226  summit=195  center=18803031
window shape: (4, 500)   sum per base: [1. 1. 1. 1. 1.] ...
bases covered (nonzero columns): 500 / 500

reconstructed (first 80): GCAGGTCCCCGGCAGCAGGAATGTGATGATGTAAGCAACAAGTGCGCCCTCTGGTGCCATATATAAAACTTGCGAAAAGG
N count in window: 0


## 4. Build the full dataset

This runs the whole pipeline (positives + negatives) and caches the result to `data/processed/dataset_w{window}_n{neg_per_pos}_s{seed}.npz`. On the first run it takes a few minutes (the genome is 3 GB); after that it loads from cache in under a second.

**What to verify below:**
- `X_pos` and `X_neg` both have shape `(N, 4, 500)` and dtype `float32`
- `chroms_pos` and `chroms_neg` are `object` (arrays of strings), shape `(N,)`

If `chroms_*` are missing, you have a stale cache from an earlier version of `src/data.py` — delete `data/processed/*.npz` and re-run.

In [9]:
data = build_dataset(cfg)

for k, v in data.items():
    print(f"{k}: shape={v.shape}  dtype={v.dtype}")

X_pos: shape=(45725, 4, 500)  dtype=float32
chroms_pos: shape=(45725,)  dtype=object
X_neg: shape=(45725, 4, 500)  dtype=float32
chroms_neg: shape=(45725,)  dtype=object
